# Lab 05 — AI_EXTRACT & AI_FILTER

**Extract** pulls structured fields from unstructured text. **Filter** acts as a semantic WHERE clause.

| Function | Returns | Use Case |
|---|---|---|
| `AI_EXTRACT` | JSON with named fields | Pull structure from logs, tickets, emails |
| `AI_FILTER` | TRUE/FALSE | Semantic filtering without regex or LIKE |


## Step 1 — Environment Setup

> **AI SQL Functions** — This lab uses the preferred `AI_` prefixed functions. 
 `AI_EXTRACT`, `AI_FILTER` — new, no legacy equivalent.

In [ ]:
USE ROLE DS_ROLE;
USE WAREHOUSE DS_WH;

CREATE DATABASE IF NOT EXISTS GENAI_LEARNING;
CREATE SCHEMA IF NOT EXISTS GENAI_LEARNING.PUBLIC;
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

## Step 2 — AI_EXTRACT: Pull Structure from Logs

In [ ]:
CREATE OR REPLACE TABLE web_server_logs (
    log_id INT AUTOINCREMENT, raw_log VARIANT
);

INSERT INTO web_server_logs (raw_log) VALUES
(PARSE_JSON('{"ip":"192.168.1.42","method":"GET","path":"/api/v2/users","status":200,"bytes":1420,"user_agent":"Mozilla/5.0","response_time_ms":45}')),
(PARSE_JSON('{"ip":"10.0.3.15","method":"POST","path":"/api/v2/orders","status":201,"bytes":890,"user_agent":"Python-Requests/2.28","response_time_ms":120}')),
(PARSE_JSON('{"ip":"10.0.1.55","method":"PUT","path":"/api/v2/users/profile","status":500,"bytes":340,"user_agent":"Mobile-App/3.2","response_time_ms":5200}')),
(PARSE_JSON('{"ip":"10.0.0.1","method":"POST","path":"/api/v2/auth/login","status":401,"bytes":120,"user_agent":"Mozilla/5.0","response_time_ms":33}')),
(PARSE_JSON('{"ip":"172.16.5.20","method":"GET","path":"/api/v2/reports/daily","status":200,"bytes":45000,"user_agent":"Tableau/2023.3","response_time_ms":1500}')),
(PARSE_JSON('{"ip":"192.168.3.75","method":"POST","path":"/api/v2/uploads","status":413,"bytes":0,"user_agent":"Chrome/119.0","response_time_ms":50}'));

In [ ]:
SELECT
    log_id,
    raw_log:status::INT AS http_status,
    AI_EXTRACT(
        raw_log::STRING,
        ['endpoint_category', 'is_error', 'client_type']
    ) AS extracted
FROM web_server_logs;

In [ ]:
WITH extracted AS (
    SELECT
        raw_log:status::INT AS status_code,
        raw_log:response_time_ms::INT AS response_time_ms,
        AI_EXTRACT(raw_log::STRING, ['error_severity', 'endpoint_category']) AS ai_fields
    FROM web_server_logs
    WHERE raw_log:status::INT >= 400
)
SELECT
    ai_fields:endpoint_category::STRING AS endpoint,
    ai_fields:error_severity::STRING AS severity,
    COUNT(*) AS error_count,
    AVG(response_time_ms) AS avg_response_ms
FROM extracted
GROUP BY endpoint, severity
ORDER BY error_count DESC;

## Step 2b — AI_EXTRACT Response Format Options

AI_EXTRACT supports multiple responseFormat styles:

| Format | Use Case |
|---|---|
| Simple object `{'key': 'question'}` | Named entity extraction |
| Array of strings `['question1', 'question2']` | Quick unnamed extraction |
| JSON schema with `type: 'string'` | Typed entity extraction |
| JSON schema with `type: 'array'` | List extraction |
| JSON schema with `type: 'object'` + `column_ordering` | Table extraction |

In [ ]:
SELECT
    log_id,
    AI_EXTRACT(
        raw_log::STRING,
        {'endpoint': 'What API endpoint was called?',
         'method': 'What HTTP method was used?',
         'status': 'What was the response status code?'}
    ) AS extracted
FROM web_server_logs
LIMIT 3;

In [ ]:
SELECT
    log_id,
    AI_EXTRACT(
        raw_log::STRING,
        {
            'schema': {
                'type': 'object',
                'properties': {
                    'endpoint': {
                        'description': 'What API endpoint was called?',
                        'type': 'string'
                    },
                    'errors': {
                        'description': 'What error messages or warnings appeared?',
                        'type': 'array'
                    }
                }
            }
        }
    ) AS structured_extract
FROM web_server_logs
LIMIT 3;

## Step 2b — AI_EXTRACT Response Format Options

AI_EXTRACT supports multiple responseFormat styles:

| Format | Use Case |
|---|---|
| Simple object `{'key': 'question'}` | Named entity extraction |
| Array of strings `['question1', 'question2']` | Quick unnamed extraction |
| JSON schema with `type: 'string'` | Typed entity extraction |
| JSON schema with `type: 'array'` | List extraction |
| JSON schema with `type: 'object'` + `column_ordering` | Table extraction |

In [ ]:
SELECT
    log_id,
    AI_EXTRACT(
        raw_log::STRING,
        {'endpoint': 'What API endpoint was called?',
         'method': 'What HTTP method was used?',
         'status': 'What was the response status code?'}
    ) AS extracted
FROM web_server_logs
LIMIT 3;

In [ ]:
SELECT
    log_id,
    AI_EXTRACT(
        raw_log::STRING,
        {
            'schema': {
                'type': 'object',
                'properties': {
                    'endpoint': {
                        'description': 'What API endpoint was called?',
                        'type': 'string'
                    },
                    'errors': {
                        'description': 'What error messages or warnings appeared?',
                        'type': 'array'
                    }
                }
            }
        }
    ) AS structured_extract
FROM web_server_logs
LIMIT 3;

## Step 3 — AI_FILTER: Semantic WHERE Clause

Describe your filter condition in plain English — no regex needed.

In [ ]:
CREATE OR REPLACE TABLE customer_feedback (
    feedback_id INT AUTOINCREMENT, product_name VARCHAR(200),
    channel VARCHAR(50), feedback VARIANT
);

INSERT INTO customer_feedback (product_name, channel, feedback) VALUES
('Mobile App', 'in-app', PARSE_JSON('{"text":"The new checkout flow is confusing and I lost my cart items twice","rating":2,"user_tier":"premium"}')),
('Mobile App', 'in-app', PARSE_JSON('{"text":"Love the new dark mode! Finally easy on the eyes at night","rating":5,"user_tier":"free"}')),
('Mobile App', 'email', PARSE_JSON('{"text":"App crashes every time I try to upload a photo","rating":1,"user_tier":"premium"}')),
('Web Dashboard', 'support', PARSE_JSON('{"text":"The export to CSV feature is broken since last update","rating":1,"user_tier":"enterprise"}')),
('Web Dashboard', 'survey', PARSE_JSON('{"text":"Great improvement in loading speed. Charts render much faster now","rating":4,"user_tier":"premium"}')),
('API Platform', 'email', PARSE_JSON('{"text":"Rate limiting is too aggressive. Our batch jobs keep failing","rating":2,"user_tier":"enterprise"}')),
('API Platform', 'support', PARSE_JSON('{"text":"Documentation is excellent. Integrated in less than a day","rating":5,"user_tier":"premium"}')),
('API Platform', 'in-app', PARSE_JSON('{"text":"Getting timeout errors on the /search endpoint during peak hours","rating":1,"user_tier":"enterprise"}'));

## Step 3b — AI_FILTER with PROMPT()

The `PROMPT()` function provides cleaner multi-column formatting for AI_FILTER predicates, using `{0}`, `{1}` placeholders.

In [ ]:
SELECT
    feedback_id,
    product_name,
    feedback:text::STRING AS text
FROM customer_feedback
WHERE AI_FILTER(
    PROMPT('This feedback about {0} describes a bug or technical issue: {1}',
           product_name,
           feedback:text::STRING)
);

In [ ]:
SELECT feedback_id, product_name, feedback:text::STRING AS text
FROM customer_feedback
WHERE AI_FILTER(feedback:text::STRING, 'describes a software bug or technical malfunction');

In [ ]:
SELECT feedback_id, product_name, feedback:text::STRING AS text
FROM customer_feedback
WHERE AI_FILTER(feedback:text::STRING, 'contains a feature request or product suggestion');

In [ ]:
SELECT feedback_id, product_name, feedback:user_tier::STRING AS tier, feedback:text::STRING AS text
FROM customer_feedback
WHERE feedback:user_tier::STRING = 'enterprise'
  AND AI_FILTER(feedback:text::STRING, 'expresses urgency, frustration, or reports a service outage');

## Step 3b — AI_FILTER with PROMPT()

The `PROMPT()` function provides cleaner multi-column formatting for AI_FILTER predicates, using `{0}`, `{1}` placeholders.

In [ ]:
SELECT
    feedback_id,
    product_name,
    feedback:text::STRING AS text
FROM customer_feedback
WHERE AI_FILTER(
    PROMPT('This feedback about {0} describes a bug or technical issue: {1}',
           product_name,
           feedback:text::STRING)
);

## Key Takeaways

- `AI_EXTRACT` infers field meanings from context — use descriptive field names.
- `AI_FILTER` acts as a semantic WHERE clause — describe conditions in plain English.
- Combine AI functions with traditional SQL filters for precise targeting.
- Both work on any text column — logs, tickets, emails, feedback.
